In [1]:
import sys; sys.path.append('../3rdparty/ElasticKnots/3rdparty/ElasticRods/python')
import sys; sys.path.append('../3rdparty/ElasticKnots/python')
import elastic_rods, elastic_knots
import numpy as np, matplotlib.pyplot as plt, time, io, os
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh

from helpers import *
from parametric_curves import *
import py_newton_optimizer 

from linkage_vis import LinkageViewer as Viewer, CenterlineViewer
from tri_mesh_viewer import PointCloudViewer, PointCloudMesh

%load_ext autoreload
%autoreload 2

import parallelism
parallelism.set_max_num_tbb_threads(1)

from MEP import MEP

Failed to load offscreen viewer: Could not load compiled module; is OffscreenRenderer missing a dependency?


In [2]:
start_file = "../data/L400-r0.2-UpTo9Crossings/4_1/0033.obj"
goal_file = "../data/L400-r0.2-UpTo9Crossings/4_1/0001.obj"
mep = MEP(start_file,goal_file)

In [3]:
cp = mep.getContactProblem()
H = cp.hessian()

triplets = list(H.entries())

rows = np.array([t.i for t in triplets])
cols = np.array([t.j for t in triplets])
vals = np.array([t.v for t in triplets])

# Matrixgröße aus den Indizes ableiten
n_rows = int(rows.max()) + 1
n_cols = int(cols.max()) + 1

# Sparse-Matrix aufbauen
H_sparse = coo_matrix((vals, (rows, cols)), shape=(n_rows, n_cols)).tocsr()

# Kleinster Eigenwert & Eigenvektor (negativer Modus)
eigvals, eigvecs = eigsh(H_sparse, k=1, which='SA')

min_eigval = eigvals[0]
Nhat = eigvecs[:, 0] / np.linalg.norm(eigvecs[:, 0])

print(f"Kleinster Eigenwert: {min_eigval}")
print(f"Nhat shape: {Nhat.shape}")


Kleinster Eigenwert: -1841.307267445252
Nhat shape: (401,)


# set dimers

In [4]:
R = cp.getDoFs() #midpoint of dimer
N = np.random.rand(len(R)) 
Nhat = N / np.linalg.norm(N) # random unit vector
delta = 0.01 #distance of R1 and R2
R1 = R + delta * Nhat
R2 = R - delta * Nhat

# get the Energy for R1 and R2

In [5]:
def getForce(R):
    cp.setDoFs(R)
    return - cp.gradient()


F1 = getForce(R1)
F2 = getForce(R2)
F0 = (F1 + F2) /2 #mean force

# compute the curvature  C

In [6]:
def curvature (F1, F2, Nhat, delta):
    C = np.dot((F2 - F1),Nhat ) / (2 * delta)
    return C

C = curvature(F1,F2,Nhat,delta)

In [7]:
def perpForce(F1, F2, Nhat):
    F1_perp = F1 - np.dot(F1, Nhat) * Nhat
    F2_perp = F2 - np.dot(F2, Nhat) * Nhat
    F_perp = F1 - F2
    theta = F_perp / np.linalg.norm(F_perp)
    return F_perp, theta 

F_perp, theta = perpForce(F1, F2, Nhat)
F_par = np.dot(F0, Nhat)* Nhat

# Rotate the dimer
First Rotate and then estimate the best rotaion


In [8]:
def rotateDimer(angle, R,Nhat, theta, delta):
    R1 = R + (Nhat* np.cos(angle) + theta * np.sin(angle))*delta
    N = R1 - R 
    Nhat = N / np.linalg.norm(N)

    R1 = R + delta * Nhat
    R2 = R - delta * Nhat
    return R1, R2, Nhat

def scalarRotationalForce(F_perp, theta, delta):
    F = np.dot(F_perp, theta)/delta 
    return F

def dScalarRotationalForce(F_star, theta_star,F, theta, d_angle):
    dF = (np.dot(F_star, theta_star) - np.dot(F, theta))/d_angle
    return dF


def rotationEstimate(F, theta, F_star, theta_star, dF):
    angle = - (np.dot(F, theta) + np.dot(F_star, theta_star) )/ (2* dF)
    return angle

In [9]:
def estimateAngle(R, Nhat,theta, delta, angle = 0.1):
    
    R1_star, R2_star , Nhat_star = rotateDimer(angle,R, Nhat, theta, delta)
    
    F1_star = getForce(R1_star)
    F2_star = getForce(R2_star)
    F_star = F1_star - F2_star / 2

    F_perp_star, theta_star = perpForce(F1_star, F2_star, Nhat_star)

    F = scalarRotationalForce(F_perp, theta, delta)
    dF = dScalarRotationalForce(F_star, theta_star,F0, theta, angle)
    angle_est = rotationEstimate(F0, theta, F_star, theta_star, dF)

    return angle_est




In [10]:
#Climb towards saddlepoint

In [11]:
def getEffectiveForce(angle_est, R, Nhat, theta, delta):
    R1, R2 , Nhat = rotateDimer(angle_est,R, Nhat, theta, delta)
    F1 = getForce(R1)
    F2 = getForce(R2)
    F0 = F1 - F2 / 2
    Nhat = Nhat / np.linalg.norm(Nhat)
    F_eff = F0 - 2* np.dot(F0, Nhat)* Nhat
    return F_eff, Nhat

In [12]:
view = Viewer(mep.getRodList(), width=1024, height=800)
mep.setViewer(view)
view.show()

Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [13]:
R = cp.getDoFs() #midpoint of dimer
N = np.random.rand(len(R)) 
Nhat = N / np.linalg.norm(N) # random unit vector
delta = 0.1 #distance of R1 and R2
for i in range(100001):
    
    
    R1 = R + delta * Nhat
    R2 = R - delta * Nhat
    
    
    F1 = getForce(R1)
    F2 = getForce(R2)
    F0 = (F1 + F2) /2 #mean force
    
    F_perp, theta = perpForce(F1, F2, Nhat)
    F_par = np.dot(F0, Nhat)* Nhat
    
    angle = estimateAngle(R, Nhat,theta, delta)
    F_eff, Nhat = getEffectiveForce(angle, R, Nhat, theta, delta)
    
    #step = min(1e-2,(np.linalg.norm(F_eff) + 1e-8))
    step = 0.01
    R = R + step * F_eff
    print(f"Contact Energy {cp.contactEnergy()}")
    cp.setDoFs(R)
    if(i % 10 == 0):
        view.update()
        print(f"it {i}, with step: {step}, F_eff: {np.linalg.norm(F_eff)} and Knot energy: {cp.energy()}")

Contact Energy 10.366298797935293
it 0, with step: 0.01, F_eff: 203.19351310563445 and Knot energy: 12.005540702475395
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
it 10, with step: 0.01, F_eff: 2.4347604812132735 and Knot energy: 2.105446569341031
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
it 20, with step: 0.01, F_eff: 1.3171481252534525 and Knot energy: 1.5135370428614057
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0.0
it 30, with step: 0.01, F_eff: 0.9060332098528439 and Knot energy: 1.2853597630348892
Contact Energy 0.0
Contact Energy 0.0
Contact Energy 0